In [8]:
from __future__ import annotations

import logging
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

MINIO_ENDPOINT = "http://urbangreen-minio:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin123")
MINIO_BUCKET = os.getenv("MINIO_STAGING_BUCKET", "staging")

SENSOR_SCHEMA = StructType(
    [
        StructField("farm_sensor_id", IntegerType(), nullable=False),
        StructField("farm_id", IntegerType(), nullable=False),
        StructField("sensor_type_id", IntegerType(), nullable=False),
        StructField("value", DoubleType(), nullable=False),
        StructField("timestamp", LongType(), nullable=False),
    ]
)

logger = logging.getLogger(__name__)

In [9]:
def env(name: str, default: str) -> str:
    return os.environ.get(name, default)


def build_spark_session() -> SparkSession:
    minio_api_port = env("MINIO_API_PORT", "9000")
    minio_endpoint = f"http://urbangreen-minio:{minio_api_port}"
    minio_access_key = env("MINIO_ROOT_USER", "minioadmin")
    minio_secret_key = env("MINIO_ROOT_PASSWORD", "minioadmin123")

    return (
        SparkSession.builder
        .appName("sensor-readings-stream-notebook-test")
        .master("spark://urbangreen-spark-master:7077")
        .config("spark.driver.host", "urbangreen-jupyter")
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", "7078")
        .config("spark.blockManager.port", "7079")
        .config("spark.executor.memory", "512m")
        .config("spark.executor.cores", "1")
        .config("spark.cores.max", "2")
    
        # Kafka connector
        .config(
            "spark.jars.packages",
            ",".join([
                "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
                "org.apache.hadoop:hadoop-aws:3.4.1",
            ])
        )
    
        # MinIO/S3A config
        .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
        )
        .getOrCreate()
    )

In [10]:
def main() -> None:
    kafka_bootstrap_servers = env(
        "SIMULATOR_KAFKA_BOOTSTRAP",
        "urbangreen-kafka:9092",
    )
    kafka_topic = env(
        "KAFKA_TOPIC_SENSOR_READINGS",
        "sensor_readings",
    )
    kafka_starting_offsets = env("KAFKA_STARTING_OFFSETS", "earliest")

    minio_staging_bucket = env("MINIO_STAGING_BUCKET", "staging")
    trigger_interval = env("STREAM_TRIGGER_INTERVAL", "60 seconds")

    output_path = f"s3a://{minio_staging_bucket}/raw/kafka/{kafka_topic}/"
    checkpoint_path = f"s3a://{minio_staging_bucket}/_checkpoints/kafka/{kafka_topic}/"

    spark = build_spark_session()
    spark.sparkContext.setLogLevel(env("SPARK_LOG_LEVEL", "INFO"))

    kafka_stream = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
        .option("subscribe", kafka_topic)
        .option("startingOffsets", kafka_starting_offsets)
        .option("failOnDataLoss", "false")
        .load()
    )

    parsed_stream = (
        kafka_stream.select(
            from_json(col("value").cast("string"), SENSOR_SCHEMA).alias("payload")
        )
        .select("payload.*")
        .withColumn("event_date", to_date(from_unixtime(col("timestamp"))))
    )

    query = (
        parsed_stream.writeStream.format("parquet")
        .option("path", output_path)
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .partitionBy("event_date")
        .trigger(processingTime=trigger_interval)
        .start()
    )

    try:
        query.awaitTermination()
    except Exception as e:
        logger.error(f"Streaming query failed: {e}")
        raise
    finally:
        spark.stop()

In [11]:
if __name__ == "__main__":
    main()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4da93b13-ae84-499c-b593-9586fcc833ab;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.cod

StreamingQueryException: [STREAM_FAILED] Query [id = 3eb653bf-08c5-40b4-94e1-12efc7a0352c, runId = ab10aff5-743e-4890-a3b1-2a00b039246b] terminated with exception: Multiple streaming queries are concurrently using s3a://staging/_checkpoints/kafka/sensor_readings/offsets. SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 3eb653bf-08c5-40b4-94e1-12efc7a0352c, runId = ab10aff5-743e-4890-a3b1-2a00b039246b]
Current Committed Offsets: {KafkaV2[Subscribe[sensor_readings]]: {"sensor_readings":{"0":250857,"1":193644,"2":215649}}}
Current Available Offsets: {KafkaV2[Subscribe[sensor_readings]]: {"sensor_readings":{"0":251028,"1":193776,"2":215796}}}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 FileSink[s3a://staging/raw/kafka/sensor_readings], 3eb653bf-08c5-40b4-94e1-12efc7a0352c, [path=s3a://staging/raw/kafka/sensor_readings/, checkpointLocation=s3a://staging/_checkpoints/kafka/sensor_readings/], Append
+- ~Project [farm_sensor_id#15, farm_